# Imports

In [1]:
import numpy as np
from tqdm import tqdm
import pandas as pd
import os

# Gensim
import gensim
from sklearn.feature_extraction.text import CountVectorizer

# spacy
import spacy

# sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

# gensim
from gensim.models import Word2Vec

# Word Representations

### Preprocessing

Note: The functions `lemmatization` and `gen_words` were obtained from Tutorial 3: Topic Modelling, Similarity Metrics and LDA.

In [2]:
# Reducing words to their root form (lemmas - refer to chapter 6 of the book)
def lemmatization(texts, allowed_postags = ["NOUN", "ADJ", "VERB", "ADV"]):
    """
    Perform lemmatization on a collection of texts, keeping only specified parts of speech.

    Parameters
    ----------
    texts : list
        A list of strings where each string is a document/text to be lemmatized.
    allowed_postags : list, optional
        A list of strings specifying which parts of speech to keep.
        Default is ["NOUN", "ADJ", "VERB", "ADV"].

    Returns
    -------
    list
        A list of strings where each string contains the lemmatized words
        from the corresponding input text, filtered by part of speech
        and joined with spaces.

    Examples
    --------
    texts = ["I am running quickly to the store"]
    lemmatization(texts)
    ['run quick store']  # Only keeps lemmatized NOUN/ADJ/VERB/ADV
    """

    # We'll be using the spacy library to do the lemmatization
    nlp = spacy.load("en_core_web_sm", disable = ["parser", "ner"])
    texts_out = []
    
    # Process all the texts/docs that you have at once
    for text in tqdm(texts):
        doc = nlp(text)
        new_text = []
        
        for token in doc:
            # Determine if the token part-of-speech is one the things that we specified in "allowed_pos"
            if token.pos_ in allowed_postags:
                # If the token part-of-speech is among the categories which we've specified, then append its lemma (root of the word) to the next_text list 
                new_text.append(token.lemma_)
        # Once you've gone over all the tokens that were part of the doc that you were analyzing, simply attach all the extracted words together (with space in between them) to form this new sentence containing only roots (lemmas) 
        final = " ".join(new_text)
        
        # append the newly created word to the list of processed texts       
        texts_out.append(final)
    
    return texts_out


def gen_words(texts):
    """
    Preprocess a list of texts into tokenized words using Gensim's simple_preprocess.
    
    Parameters:
        texts (list): A list of strings, where each string is a document/text to be processed
    
    Returns:
        list: A list of lists, where each inner list contains preprocessed tokens (words) from the corresponding input text
    
    Example:
        texts = ["Hello, World!", "This is a test-case."]
        gen_words(texts)
        [['hello', 'world'], ['this', 'is', 'a', 'test', 'case']]
    """
    # Preprocesses a list of texts by converting to lowercase, removing punctuation/numbers, and tokenizing into words
    final = []
    for text in texts:
        new = gensim.utils.simple_preprocess(text, deacc=True)
        final.append(new)
        
    return final

### Bag of Words Function

In [3]:
def get_bag_of_words(corpus, bBinary=False):
    if (bBinary):
        vectorizer = CountVectorizer(binary=True)
    else:
        vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(corpus)

    bag_of_words_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out()) # convert to dataFrame
    return bag_of_words_df

### Data 

In [12]:
data_dir = os.getcwd()
number_of_rows = 5000

# get plot descriptions
df = pd.read_csv("../data/output.csv", nrows=number_of_rows)
plot_descriptions = df["plot"].tolist()
genres = df["genre"].tolist()

# lemmatize plot descriptions
lemmatized_plot_descriptions = lemmatization(plot_descriptions)

# tokenize plot descriptions
tokenized_plot_descriptions = gen_words(lemmatized_plot_descriptions)

# create bag of words representation for each plot description
bag_of_words_plots = get_bag_of_words(lemmatized_plot_descriptions)
bag_of_words_plots[0:5]

100%|██████████| 5000/5000 [00:22<00:00, 225.61it/s]


,1000,10th,11th,12th,13th,14th,15th,1600,168th,16th,...,zealot,zealous,zebra,zero,zip,zombie,zone,zoo,zookeeper,zoologist
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Trained Model
- Logistic Regression Model
- Skip-Gram Word Embeddings

In [13]:
## -- Create Model -- ##
# write out model params
model_params = {
    'sg': 1,       
    'vector_size': 200, 
    'window': 5,     
    'min_count': 4, 
    'workers': 4,
    'seed': 42,
    'epochs': 30,
}
w2v_model = Word2Vec(sentences=tokenized_plot_descriptions, sg=model_params['sg'], vector_size=model_params['vector_size'], window=model_params['window'], min_count=model_params['min_count'], workers=model_params['workers'], seed=model_params['seed'], epochs=model_params['epochs'])

## -- Get Doc Embeddings Vectors -- ##
def get_doc_vectors(tokens, model):
    doc_vecs = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    if not doc_vecs:
        return np.zeros(model.vector_size)
    return np.mean(doc_vecs, axis=0)

## -- Get X and y data -- ## 
X_w2v = np.array([get_doc_vectors(texts, w2v_model) for texts in tokenized_plot_descriptions])
y_w2v = np.array(genres)
print(f"size of X: {X_w2v.shape}, size of y: {y_w2v.shape}")

## -- Train/Test Split -- ##
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(X_w2v, y_w2v, test_size=0.2, random_state=42)

## -- Train Classifier -- ##
w2v_model = LogisticRegression(max_iter=1000)
w2v_model.fit(X_train_w2v, y_train_w2v)

## -- Evaluate -- ##
y_pred_w2v = w2v_model.predict(X_test_w2v) 
print(classification_report(y_test_w2v, y_pred_w2v)) 
print(f"Macro f1 Score: {f1_score(y_test_w2v, y_pred_w2v, average='weighted')}")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


size of X: (5000, 200), size of y: (5000,)
              precision    recall  f1-score   support

      Action       0.56      0.61      0.58       160
   Adventure       0.35      0.21      0.27        52
   Animation       0.00      0.00      0.00        15
    Children       0.00      0.00      0.00        25
      Comedy       0.46      0.58      0.51       281
       Crime       0.50      0.19      0.28        63
 Documentary       0.72      0.48      0.58        27
       Drama       0.47      0.58      0.52       275
     Fantasy       0.00      0.00      0.00         2
   Film-Noir       0.00      0.00      0.00         1
      Horror       0.43      0.39      0.41        57
     Musical       0.00      0.00      0.00         3
     Mystery       0.00      0.00      0.00         6
     Romance       0.00      0.00      0.00         6
      Sci-Fi       0.00      0.00      0.00         7
    Thriller       0.00      0.00      0.00        14
         War       0.00      0.00     

c:\Users\nourf\anaconda3\envs\python4al3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nourf\anaconda3\envs\python4al3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nourf\anaconda3\envs\python4al3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

# Simple Baselines

## Majority Baseline

In [14]:
most_common_genre = df['genre'].mode()[0]
y_pred_baseline = [most_common_genre] * len(y_test_w2v)
print(classification_report(y_test_w2v, y_pred_baseline))
print(f"Macro f1 Score: {f1_score(y_test_w2v, y_pred_baseline, average='weighted')}")

              precision    recall  f1-score   support

      Action       0.00      0.00      0.00       160
   Adventure       0.00      0.00      0.00        52
   Animation       0.00      0.00      0.00        15
    Children       0.00      0.00      0.00        25
      Comedy       0.00      0.00      0.00       281
       Crime       0.00      0.00      0.00        63
 Documentary       0.00      0.00      0.00        27
       Drama       0.28      1.00      0.43       275
     Fantasy       0.00      0.00      0.00         2
   Film-Noir       0.00      0.00      0.00         1
      Horror       0.00      0.00      0.00        57
     Musical       0.00      0.00      0.00         3
     Mystery       0.00      0.00      0.00         6
     Romance       0.00      0.00      0.00         6
      Sci-Fi       0.00      0.00      0.00         7
    Thriller       0.00      0.00      0.00        14
         War       0.00      0.00      0.00         2
     Western       0.00    

c:\Users\nourf\anaconda3\envs\python4al3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nourf\anaconda3\envs\python4al3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nourf\anaconda3\envs\python4al3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is